In [1]:
import sys
from pathlib import Path

root_dir = Path().absolute()
# Strip ~/notebooks/ccfraud from PYTHON_PATH if notebook started in one of these subdirectories
if root_dir.parts[-1:] == ('notebooks',):
    root_dir = Path(*root_dir.parts[:-1])
    sys.path.append(str(root_dir))
if root_dir.parts[-1:] == ('airquality',):
    root_dir = Path(*root_dir.parts[:-1])
    sys.path.append(str(root_dir))
root_dir = str(root_dir) 

print(f"Root dir: {root_dir}")

# Set the environment variables from the file <root_dir>/.env
from mlfs import config
settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

Root dir: /hopsfs/Users/meb10000/mlfs-book


HopsworksSettings initialized!


<span style="font-width:bold; font-size: 3rem; color:#333;">- Part 02: Daily Feature Pipeline for Air Quality (aqicn.org) and weather (openmeteo)</span>

## 🗒️ This notebook is divided into the following sections:
1. Download and Parse Data
2. Feature Group Insertion


__This notebook should be scheduled to run daily__

In the book, we use a GitHub Action stored here:
[.github/workflows/air-quality-daily.yml](https://github.com/featurestorebook/mlfs-book/blob/main/.github/workflows/air-quality-daily.yml)

However, you are free to use any Python Orchestration tool to schedule this program to run daily.

### <span style='color:#ff5f27'> 📝 Imports

In [2]:
import datetime
import time
import requests
import pandas as pd
import hopsworks
from airquality import util
from mlfs import config
import json
import warnings
warnings.filterwarnings("ignore")

2026-06-26 14:42:37,643 INFO: generated new fontManager


## <span style='color:#ff5f27'> 🌍 Get the Sensor URL, Country, City, Street names from Hopsworks </span>

__Update the values in the cell below.__

__These should be the same values as in notebook 1 - the feature backfill notebook__


In [3]:
project = hopsworks.login()
fs = project.get_feature_store() 
secrets = hopsworks.get_secrets_api()

# This line will fail if you have not registered the AQICN_API_KEY as a secret in Hopsworks
AQICN_API_KEY = secrets.get_secret("AQICN_API_KEY").value
location_str = secrets.get_secret("SENSOR_LOCATION_JSON").value
location = json.loads(location_str)

country=location['country']
city=location['city']
street=location['street']
aqicn_url=location['aqicn_url']
latitude=location['latitude']
longitude=location['longitude']

today = datetime.date.today()

location_str

2026-06-26 14:42:40,729 INFO: Python Engine initialized.



Logged in to project, explore it here https://hopsworks.ai.local/p/122


'{"country": "Sweden", "city": "Stockholm", "street": "Hornsgatan-108", "aqicn_url": "https://api.waqi.info/feed/@10009", "latitude": 59.33, "longitude": 18.07}'

### <span style="color:#ff5f27;"> 🔮 Get references to the Feature Groups </span>

In [4]:
# Retrieve feature groups
air_quality_fg = fs.get_feature_group(
    name='air_quality',
    version=1,
)
weather_fg = fs.get_feature_group(
    name='weather',
    version=1,
)

---

## <span style='color:#ff5f27'> 🌫 Retrieve Today's Air Quality data (PM2.5) from the AQI API</span>


In [5]:
import requests
import pandas as pd

aq_today_df = util.get_pm25(aqicn_url, country, city, street, today, AQICN_API_KEY)
aq_today_df

,pm25,country,city,street,date,url
0,9.0,Sweden,Stockholm,Hornsgatan-108,2026-06-26,https://api.waqi.info/feed/@10009


In [6]:
aq_today_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   pm25     1 non-null      float32       
 1   country  1 non-null      object        
 2   city     1 non-null      object        
 3   street   1 non-null      object        
 4   date     1 non-null      datetime64[ns]
 5   url      1 non-null      object        
dtypes: datetime64[ns](1), float32(1), object(4)
memory usage: 176.0+ bytes


## <span style='color:#ff5f27'> 🌦 Get Weather Forecast data</span>

In [7]:
hourly_df = util.get_hourly_weather_forecast(city, latitude, longitude)
hourly_df = hourly_df.set_index('date')

# We will only make 1 daily prediction, so we will replace the hourly forecasts with a single daily forecast
# We only want the daily weather data, so only get weather at 12:00
daily_df = hourly_df.between_time('11:59', '12:01')
daily_df = daily_df.reset_index()
daily_df['date'] = pd.to_datetime(daily_df['date']).dt.date
daily_df['date'] = pd.to_datetime(daily_df['date'])
daily_df['city'] = city
daily_df

Coordinates 59.25°N 18.0°E
Elevation 18.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s


,date,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant,city
0,2026-06-26,23.462500,0.0,21.758419,196.336121,Stockholm
1,2026-06-27,29.812500,0.0,8.707237,240.255203,Stockholm
2,2026-06-28,31.062500,0.0,10.464798,266.054901,Stockholm
3,2026-06-29,28.162500,0.0,16.919998,270.000000,Stockholm
4,2026-06-30,24.362501,0.0,9.178235,64.440094,Stockholm
5,2026-07-01,22.912500,0.0,15.256526,70.709862,Stockholm
6,2026-07-02,23.812500,0.0,4.582052,135.000107,Stockholm


In [8]:
daily_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 6 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   date                         7 non-null      datetime64[ns]
 1   temperature_2m_mean          7 non-null      float32       
 2   precipitation_sum            7 non-null      float32       
 3   wind_speed_10m_max           7 non-null      float32       
 4   wind_direction_10m_dominant  7 non-null      float32       
 5   city                         7 non-null      object        
dtypes: datetime64[ns](1), float32(4), object(1)
memory usage: 356.0+ bytes


## <span style="color:#ff5f27;">⬆️ Uploading new data to the Feature Store</span>

In [9]:
# Insert new data
air_quality_fg.insert(aq_today_df)

2026-06-26 14:42:46,743 INFO: Created temporary directory '/tmp/tmphhbgi1vm' for ephemeral docs site


2026-06-26 14:42:46,744 INFO: Loading 'datasources' ->
[]


Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

Calculating Metrics:  50%|█████     | 2/4 [00:00<00:00, 3111.50it/s]

...

Calculating Metrics: 100%|██████████| 4/4 [00:00<00:00, 1110.63it/s]

Validation succeeded.


Validation Report saved successfully, explore a summary at https://hopsworks.ai.local/p/122/fs/70/fg/2069


2026-06-26 14:42:56,712 INFO: Computing insert statistics


(None,
 {
   "success": true,
   "results": [
     {
       "success": true,
       "expectation_config": {
         "kwargs": {
           "batch_id": "hopsworks_pandas-hopsworks_asset",
           "column": "pm25",
           "min_value": -0.1,
           "max_value": 500.0,
           "strict_min": true
         },
         "meta": {
           "expectationId": 1027
         }
       },
       "result": {
         "observed_value": 9.0
       },
       "meta": {
         "ingestionResult": "INGESTED",
         "validationTime": "2026-06-26T12:42:46.000826Z"
       },
       "exception_info": {
         "raised_exception": false,
         "exception_traceback": null,
         "exception_message": null
       }
     }
   ],
   "suite_name": "",
   "suite_parameters": {},
   "statistics": {
     "evaluated_expectations": 1,
     "successful_expectations": 1,
     "unsuccessful_expectations": 0,
     "success_percent": 100.0
   },
   "meta": {
     "great_expectations_version": "1.17.1"

In [10]:
# Insert new data
weather_fg.insert(daily_df, wait=True)

2026-06-26 14:42:57,282 INFO: Created temporary directory '/tmp/tmpfg7r3i4r' for ephemeral docs site


2026-06-26 14:42:57,283 INFO: Loading 'datasources' ->
[]


Calculating Metrics:   0%|          | 0/5 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/5 [00:00<?, ?it/s]

Calculating Metrics:  40%|████      | 2/5 [00:00<00:00, 3515.76it/s]

...

Calculating Metrics: 100%|██████████| 5/5 [00:00<00:00, 1324.88it/s]

Validation succeeded.


Validation Report saved successfully, explore a summary at https://hopsworks.ai.local/p/122/fs/70/fg/2070


2026-06-26 14:43:07,153 INFO: Computing insert statistics


(None,
 {
   "success": true,
   "results": [
     {
       "success": true,
       "expectation_config": {
         "kwargs": {
           "batch_id": "hopsworks_pandas-hopsworks_asset",
           "column": "wind_speed_10m_max",
           "min_value": -0.1,
           "max_value": 1000.0,
           "strict_min": true
         },
         "meta": {
           "expectationId": 1028
         }
       },
       "result": {
         "observed_value": 4.582051753997803
       },
       "meta": {
         "ingestionResult": "INGESTED",
         "validationTime": "2026-06-26T12:42:57.000366Z"
       },
       "exception_info": {
         "raised_exception": false,
         "exception_traceback": null,
         "exception_message": null
       }
     },
     {
       "success": true,
       "expectation_config": {
         "kwargs": {
           "batch_id": "hopsworks_pandas-hopsworks_asset",
           "column": "precipitation_sum",
           "min_value": -0.1,
           "max_value": 100

## <span style="color:#ff5f27;">⏭️ **Next:** Part 03: Training Pipeline
 </span> 

In the following notebook you will read from a feature group and create training dataset within the feature store
